# Quantisation pipeline
(Using GPTQModels on NVIDIA V100)

Quantised models are stored in `_temp/` for use by `inference/project.ipynb`.
Use of Volta GPUs introduce the following constraints:
- `use_marlin=False` (marlin kernel are bound to sm $\ge$ 8.0), 
- `dtype=float16` (V100 lacks support for bit depths $\ge$ bf16), 
- triton $\le$ 2.1.0 (for triton kernels, later versions lack sm_70 support)

Calibration set $\ne$ benchmark train set for data leakage prevention.

In [ ]:
import logging
from quantisation import QuantConf

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("quant_run.log"),
        logging.StreamHandler(),
    ],
    force=True,
)

BASE_PATH = "_temp"
MODEL_SELECTION = ["qwen", "deepseek"]
QUANTISATION_BITS = [4, 8]
QUANTISATION_GROUPLEVEL = 128
BATCH_SIZE = 1

handle_map = {
    "qwen":     "Qwen/Qwen3-0.6B",
    "deepseek": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
}

In [ ]:
import itertools, torch
import datasets as hf_datasets
from tqdm.auto import tqdm

logging.info("[DATA] Loading gsm8k train for calibration")
cal_ds = hf_datasets.load_dataset("openai/gsm8k", "main", split="train")
calibration_dataset = [
    f"Problem: {row['question']}\nSolution: {row['answer']}"
    for row in cal_ds
][:128]
logging.info(f"[DATA] {len(calibration_dataset)} calibration samples ready")

params = list(itertools.product(MODEL_SELECTION, QUANTISATION_BITS))
logging.info(f"[RUN] {len(params)} quantisation configurations")

for model_name, quant_bits in tqdm(params, desc="Quant configs"):
    logging.info(f"[RUN] model={model_name}  bits={quant_bits}")
    conf = QuantConf(
        model_name=model_name,
        quant_bits=quant_bits,
        quant_groupsize=QUANTISATION_GROUPLEVEL,
        handle=handle_map[model_name],
        calibration_data=calibration_dataset,
        base_path=BASE_PATH,
    )
    conf.quantise()
    torch.cuda.empty_cache()

logging.info(f"[RUN] All models quantised and saved to {BASE_PATH}")